# 02 — Ingeniería de Features

**Objetivo:** construir el dataset final listo para entrenar la red neuronal.

**Salida:** `data/processed/train.csv`, `val.csv`, `test.csv`, `preprocesadores.pkl`

In [ ]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from sklearn.preprocessing import StandardScaler, LabelEncoder

DATA_DIR = Path('../dataset')
OUT_DIR  = Path('../data/processed')

## 1. Carga de datos

In [ ]:
orders    = pd.read_csv(DATA_DIR / 'olist_orders_dataset.csv', parse_dates=[
    'order_purchase_timestamp', 'order_delivered_customer_date',
    'order_estimated_delivery_date'
])
items     = pd.read_csv(DATA_DIR / 'olist_order_items_dataset.csv')
products  = pd.read_csv(DATA_DIR / 'olist_products_dataset.csv')
customers = pd.read_csv(DATA_DIR / 'olist_customers_dataset.csv')
sellers   = pd.read_csv(DATA_DIR / 'olist_sellers_dataset.csv')
geo       = pd.read_csv(DATA_DIR / 'olist_geolocation_dataset.csv')
cat_names = pd.read_csv(DATA_DIR / 'product_category_name_translation.csv')

## 2. Filtrar órdenes válidas y calcular targets

In [ ]:
# Solo órdenes entregadas con fecha real disponible
df = orders[
    (orders['order_status'] == 'delivered') &
    orders['order_delivered_customer_date'].notna()
].copy()

# Variables objetivo
df['dias_entrega'] = (
    df['order_delivered_customer_date'] - df['order_purchase_timestamp']
).dt.days

df['es_retraso'] = (
    df['order_delivered_customer_date'] > df['order_estimated_delivery_date']
).astype(int)

# Eliminar outliers
df = df[(df['dias_entrega'] >= 0) & (df['dias_entrega'] <= 60)]
print(f'Órdenes válidas: {len(df):,}')

## 3. Features temporales

In [ ]:
df['mes_compra']       = df['order_purchase_timestamp'].dt.month
df['dia_semana_compra'] = df['order_purchase_timestamp'].dt.dayofweek  # 0=lunes
df['hora_compra']      = df['order_purchase_timestamp'].dt.hour

## 4. Join con items + productos + sellers

In [ ]:
# Agregar por orden: sumar precio y flete, tomar el seller con mayor precio
items_agg = items.sort_values('price', ascending=False).groupby('order_id').agg(
    precio_total  = ('price', 'sum'),
    flete_total   = ('freight_value', 'sum'),
    product_id    = ('product_id', 'first'),   # producto principal
    seller_id     = ('seller_id', 'first'),    # seller principal
).reset_index()

df = df.merge(items_agg, on='order_id', how='left')

# Unir productos
products = products.merge(
    cat_names, on='product_category_name', how='left'
)
df = df.merge(
    products[['product_id','product_category_name_english',
              'product_weight_g','product_length_cm',
              'product_height_cm','product_width_cm']],
    on='product_id', how='left'
)
df.rename(columns={'product_category_name_english': 'categoria_producto'}, inplace=True)

# Volumen del producto
df['volumen_cm3'] = (
    df['product_length_cm'] * df['product_height_cm'] * df['product_width_cm']
)

# Unir sellers
df = df.merge(
    sellers[['seller_id','seller_zip_code_prefix','seller_state']],
    on='seller_id', how='left'
)

# Unir clientes
df = df.merge(
    customers[['customer_id','customer_zip_code_prefix','customer_state']],
    on='customer_id', how='left'
)

print(df.shape)

## 5. Calcular distancia Haversine

In [ ]:
# Coordenadas promedio por código postal (usar mediana para robustez)
geo_agg = geo.groupby('geolocation_zip_code_prefix').agg(
    lat=('geolocation_lat', 'median'),
    lng=('geolocation_lng', 'median')
).reset_index()

# Unir coordenadas del vendedor
df = df.merge(
    geo_agg.rename(columns={'geolocation_zip_code_prefix': 'seller_zip_code_prefix',
                            'lat': 'seller_lat', 'lng': 'seller_lng'}),
    on='seller_zip_code_prefix', how='left'
)

# Unir coordenadas del cliente
df = df.merge(
    geo_agg.rename(columns={'geolocation_zip_code_prefix': 'customer_zip_code_prefix',
                            'lat': 'customer_lat', 'lng': 'customer_lng'}),
    on='customer_zip_code_prefix', how='left'
)

def haversine(lat1, lng1, lat2, lng2):
    """Distancia en km entre dos puntos geográficos."""
    R = 6371.0
    lat1, lng1, lat2, lng2 = map(np.radians, [lat1, lng1, lat2, lng2])
    dlat = lat2 - lat1
    dlng = lng2 - lng1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlng/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

df['distancia_km'] = haversine(
    df['seller_lat'],  df['seller_lng'],
    df['customer_lat'], df['customer_lng']
)

print(f"Distancia media: {df['distancia_km'].mean():.1f} km")
print(f"Nulos en distancia_km: {df['distancia_km'].isna().sum()}")

## 6. Seleccionar columnas y tratar nulos

In [ ]:
FEATURES_NUM = [
    'distancia_km', 'precio_total', 'flete_total',
    'product_weight_g', 'volumen_cm3',
    'mes_compra', 'dia_semana_compra', 'hora_compra'
]
FEATURES_CAT = ['categoria_producto', 'customer_state', 'seller_state']
TARGETS      = ['dias_entrega', 'es_retraso']

df_modelo = df[FEATURES_NUM + FEATURES_CAT + TARGETS + ['order_purchase_timestamp']].copy()

# Nulos en columnas numéricas → mediana
for col in FEATURES_NUM:
    mediana = df_modelo[col].median()
    df_modelo[col] = df_modelo[col].fillna(mediana)

# Nulos en categóricas → 'desconocido'
for col in FEATURES_CAT:
    df_modelo[col] = df_modelo[col].fillna('desconocido')

print(f'Dataset limpio: {df_modelo.shape}')
print(f'Nulos restantes:\n{df_modelo.isnull().sum()[df_modelo.isnull().sum() > 0]}')

## 7. Split cronológico (80/10/10)

In [ ]:
df_modelo = df_modelo.sort_values('order_purchase_timestamp').reset_index(drop=True)
n = len(df_modelo)

n_train = int(n * 0.80)
n_val   = int(n * 0.90)

train = df_modelo.iloc[:n_train].copy()
val   = df_modelo.iloc[n_train:n_val].copy()
test  = df_modelo.iloc[n_val:].copy()

print(f'Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}')
print(f'Train hasta:  {train["order_purchase_timestamp"].max().date()}')
print(f'Val hasta:    {val["order_purchase_timestamp"].max().date()}')
print(f'Test hasta:   {test["order_purchase_timestamp"].max().date()}')

## 8. Preprocesamiento: escalado y codificación

In [ ]:
# Escalar numéricas (ajustar solo en train)
scaler = StandardScaler()
train[FEATURES_NUM] = scaler.fit_transform(train[FEATURES_NUM])
val[FEATURES_NUM]   = scaler.transform(val[FEATURES_NUM])
test[FEATURES_NUM]  = scaler.transform(test[FEATURES_NUM])

# Codificar categóricas con LabelEncoder (ajustar solo en train)
encoders = {}
for col in FEATURES_CAT:
    le = LabelEncoder()
    # Ajustar en train
    le.fit(train[col])
    # Categorías no vistas en val/test → mapear a 'desconocido' (siempre está en train)
    for split in [train, val, test]:
        split[col] = split[col].apply(
            lambda x: x if x in le.classes_ else 'desconocido'
        )
    train[col] = le.transform(train[col])
    val[col]   = le.transform(val[col])
    test[col]  = le.transform(test[col])
    encoders[col] = le
    print(f'{col}: {len(le.classes_)} categorías')

print('\nEjemplo train:')
train[FEATURES_CAT].head(2)

## 9. Guardar datos procesados y preprocesadores

In [ ]:
cols_guardar = FEATURES_NUM + FEATURES_CAT + TARGETS

train[cols_guardar].to_csv(OUT_DIR / 'train.csv', index=False)
val[cols_guardar].to_csv(OUT_DIR / 'val.csv', index=False)
test[cols_guardar].to_csv(OUT_DIR / 'test.csv', index=False)

# Guardar también los cardinalidades para los embeddings
cardinalidades = {col: len(enc.classes_) for col, enc in encoders.items()}

with open(OUT_DIR / 'preprocesadores.pkl', 'wb') as f:
    pickle.dump({'scaler': scaler, 'encoders': encoders,
                 'cardinalidades': cardinalidades,
                 'features_num': FEATURES_NUM,
                 'features_cat': FEATURES_CAT}, f)

print('Archivos guardados en data/processed/')
print(f'Cardinalidades embeddings: {cardinalidades}')